In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Malicious Prompt Detection - Hackathon Submission

**Participation Details:**
* **Full Name:** Shourya Varshney
* **College Name:** GLA University, Mathura
* **Phone Number:** 7247870237
* **Email Address:** varshneyshourya19@gmail.com
* **Google Drive Link (Model, Output & Metrics):** [Paste your public Google Drive link here]

using basic libraries for prediction of wrong one.

In [ ]:
import pandas as pd

# Load the train and test data
train_df = pd.read_csv('/kaggle/input/competitions/data-sprint-prompt-based-classififcation/merged_train_70.csv')   # Update the path!
test_df = pd.read_csv('/kaggle/input/competitions/data-sprint-prompt-based-classififcation/merged_test_30.csv')
print("--- TRAINING DATA INFO ---")
print(train_df.info())

### Smart Sampling
Here i am Spliting data in 50/50 format because giving more right data cause to do less work from model and predict that all are correct which is a nightmare.

In [ ]:
import pandas as pd

# 1. Extract the explicitly labeled malicious and safe prompts
df_labeled = train_df[['Prompt', 'isMalicious']].dropna()
df_labeled.rename(columns={'Prompt': 'text', 'isMalicious': 'label'}, inplace=True)

# 2. Extract the Quora questions (these are safe/benign, so label them 0)
df_quora = train_df[['question1']].dropna()
df_quora['label'] = 0.0
df_quora.rename(columns={'question1': 'text'}, inplace=True)

# 3. Extract the "text" column from the Malignant dataset (these are usually malicious, label 1)
df_malignant = train_df[['text', 'base_class']].dropna()
df_malignant['label'] = df_malignant['base_class'].apply(lambda x: 1.0 if str(x).lower() != 'safe' else 0.0)
df_malignant.drop(columns=['base_class'], inplace=True)

# 4. Combine everything into one master, perfectly clean dataset
clean_train = pd.concat([df_labeled, df_quora, df_malignant], ignore_index=True)

# 5. Shuffle the data and ensure labels are integers (0 or 1)
clean_train = clean_train.sample(frac=1, random_state=42).reset_index(drop=True)
clean_train['label'] = clean_train['label'].astype(int)

# Print the final shape and the balance of safe vs malicious
print("--- CLEANED DATASET ---")
print(clean_train.info())
print("\nLabel Balance:\n", clean_train['label'].value_counts())

Making some changes accordingly for getting massive f1 score as much as possible also for taking a research for epochs

In [ ]:
# Separate the classes
df_safe = clean_train[clean_train['label'] == 0]
df_malicious = clean_train[clean_train['label'] == 1]

# Undersample the safe class to exactly match the malicious class
df_safe_balanced = df_safe.sample(n=len(df_malicious), random_state=42)

# Combine them back together into a final, balanced dataset
final_train = pd.concat([df_safe_balanced, df_malicious], ignore_index=True)

# Shuffle the data one last time
final_train = final_train.sample(frac=1, random_state=42).reset_index(drop=True)

print("--- FINAL BALANCED DATASET ---")
print(final_train['label'].value_counts())
print(f"Total rows for fast training: {len(final_train)}")

**Model Insights**
Setting up transformer and more to try to get better score also for now i am working with distilbert base uncased.

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

print("Starting Transformer Setup...")

# 1. Split the balanced data (80% for training, 20% for testing our F1 score)
train_split = final_train.sample(frac=0.8, random_state=42)
val_split = final_train.drop(train_split.index)

# Convert Pandas DataFrames to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_split)
val_ds = Dataset.from_pandas(val_split)

# 2. Load the Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function (keeps max length to 128 to train incredibly fast tonight)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing data (this takes about 30 seconds)...")
train_tokenized = train_ds.map(tokenize_function, batched=True)
val_tokenized = val_ds.map(tokenize_function, batched=True)

# 3. Define the F1 Metric for the Judges
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "f1": f1_score(labels, predictions),
        "accuracy": accuracy_score(labels, predictions)
    }

# 4. Load the Pre-trained Model Structure (Allowed by T&Cs)
print("Downloading DistilBERT...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 5. Set up Training Arguments for a Fast Sprint
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",    # Check F1 score after every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=32, # Maximize GPU usage
    per_device_eval_batch_size=32,
    num_train_epochs=2,             # 2 epochs is plenty for 30k rows
    weight_decay=0.01,
    report_to="none"                # Turn off external logging to prevent Kaggle errors
)

# 6. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print("Starting Training! Grab a coffee, this will take about 15 minutes...")
# 7. TRAIN!
trainer.train()

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
import shutil
import os

print("1. Preparing Test Data...")
# Combine the scattered text columns into exactly 'text'
test_df['text'] = test_df['Prompt'].fillna(test_df['question1']).fillna(test_df['text'])

# Ensure the text is strings and keep the id
test_clean = test_df[['id', 'text']].copy()

# Fix the ID column (using the .to_series() fix)
test_clean['id'] = test_clean['id'].fillna(test_clean.index.to_series()).astype(int)
test_clean['text'] = test_clean['text'].astype(str)

print("2. Tokenizing Test Data...")
test_ds = Dataset.from_pandas(test_clean)
test_tokenized = test_ds.map(tokenize_function, batched=True)

print("3. Generating Predictions (This takes about 2-3 minutes)...")
# Run the test data through your trained model
raw_predictions = trainer.predict(test_tokenized)

# Convert the raw probabilities into 0 or 1 labels
final_labels = np.argmax(raw_predictions.predictions, axis=-1)

print("4. Saving Submission File...")
# Create the exact format required: id, label
submission_df = pd.DataFrame({
    'id': test_clean['id'],
    'label': final_labels
})

# Save to CSV
submission_filename = 'submission.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"SUCCESS! Predictions saved to {submission_filename}")

print("5. Saving and Zipping the Model...")
# Define absolute Kaggle paths
model_dir = "/kaggle/working/my_trained_model"
zip_path = "/kaggle/working/trained_model_submission"

# Save weights and tokenizer
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

# Compress the folder into a .zip file
shutil.make_archive(zip_path, 'zip', model_dir)

print(f"SUCCESS! The model zip file has been created as {zip_path}.zip")
print("\n--- ALL TASKS COMPLETE! ---")
print("Refresh your Kaggle output panel to download submission.csv and trained_model_submission.zip")

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, f1_score

print("Generating the Output File for Judges...")

# Calculate the metrics using your validation data 
# (Since the hidden test data doesn't have the answers for us to check)
val_predictions_raw = trainer.predict(val_tokenized)
val_preds = np.argmax(val_predictions_raw.predictions, axis=-1)
val_true_labels = val_split['label'].tolist()

cm = confusion_matrix(val_true_labels, val_preds)
acc = accuracy_score(val_true_labels, val_preds)
prec = precision_score(val_true_labels, val_preds)
f1 = f1_score(val_true_labels, val_preds)

# Create the dedicated text file
with open('DataSprint_Output_Metrics.txt', 'w') as f:
    f.write("--- DATASPRINT OUTPUT METRICS ---\n")
    f.write("Note: Metrics calculated on the 20% validation split.\n\n")
    f.write(f"1. F1 Score:  {f1:.4f}\n")
    f.write(f"2. Accuracy:  {acc:.4f}\n")
    f.write(f"3. Precision: {prec:.4f}\n\n")
    f.write(f"4. Confusion Matrix:\n{cm}\n")

print("SUCCESS! Metrics saved to 'DataSprint_Output_Metrics.txt'")